In [2]:
from rdflib import Graph, Namespace
import pandas as pd

ns = Namespace("http://n4d.sh/civ6#")

# Загрузите онтологию
g = Graph()
g.parse("civ6-ontology-v2.rdf", format="xml")

<Graph identifier=N82666e6375304df59bb648dcced054ac (<class 'rdflib.graph.Graph'>)>

In [3]:
def run_sparql_query(query, graph):
    """
    Executes a SPARQL query and returns results as a pandas DataFrame.
    """
    results = graph.query(query)
    data = []

    for row in results:
        data.append(
            {var: row[var] for var in row.labels}
        )  # Collect variables and their values

    return pd.DataFrame(data)

Отобразим найденных с помощью wikidata лидеров


In [4]:
query = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX : <http://n4d.sh/civ6#>

SELECT ?entity ?name ?civilizationName
WHERE {
  ?entity :source "wikidata" ;
          :name ?name .
  OPTIONAL {
    ?entity :rules ?civilization .
    ?civilization :name ?civilizationName .
  }
}
"""

run_sparql_query(query, g)

,entity,name,civilizationName
0,http://n4d.sh/civ6#Gorgo,Gorgo,Greece
1,http://n4d.sh/civ6#Gorgo,Gorgo,Sparta
2,http://n4d.sh/civ6#John_Curtin,John Curtin,Australia
3,http://n4d.sh/civ6#Victoria,Victoria,England
4,http://n4d.sh/civ6#Victoria,Victoria,United Kingdom of Great Britain and Ireland
5,http://n4d.sh/civ6#Hedwig_of_Silesia,Hedwig of Silesia,Holy Roman Empire
6,http://n4d.sh/civ6#Tomyris,Tomyris,Scythia
7,http://n4d.sh/civ6#Tomyris,Tomyris,Massagetae
8,http://n4d.sh/civ6#Hōjō_Tokimune,Hōjō Tokimune,Japan
9,http://n4d.sh/civ6#Frédéric_Barberousse,Frédéric Barberousse,Holy Roman Empire


In [5]:
query = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX : <http://n4d.sh/civ6#>

SELECT ?leader ?name
WHERE {
  ?leader a :Leader ;
          :name ?name ;
          :rules "None" .
}
"""

run_sparql_query(query, g)[0:10]

,leader,name
0,http://n4d.sh/civ6#Jayavarman_VII,Jayavarman VII
1,http://n4d.sh/civ6#Saladin,Saladin
2,http://n4d.sh/civ6#Tribhuwana_Wijayatunggadewi,Tribhuwana Wijayatunggadewi
3,http://n4d.sh/civ6#Moctezuma_I,Moctezuma I


In [8]:
query = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX : <http://n4d.sh/civ6#>

SELECT ?religion (COUNT(DISTINCT ?civilization) AS ?civilization_count)
WHERE {
  ?leader a :Leader ;
          :rules ?civilization ;
          :believesIn ?religion .
}
GROUP BY ?religion
ORDER BY DESC(?civilization_count)
"""

run_sparql_query(query, g).head(5)

,religion,civilization_count
0,http://n4d.sh/civ6#Catholicism,12
1,http://n4d.sh/civ6#Protestantism,7
2,http://n4d.sh/civ6#Buddhism,6
3,http://n4d.sh/civ6#Eastern_Orthodoxy,5
4,http://n4d.sh/civ6#Islam,5
